# 1. Energy signature: how much gas the house burns for the weather

**Ticket:** TECH-98 (ML spike), Insights epic TECH-E7.

**The question:** on a given day, how much gas should the house burn, given how cold, windy and sunny it was?

Answering that does two jobs:
1. **It describes the house** in a few physical numbers: how much gas goes on hot water, how many kWh each extra degree of cold costs, the outdoor temperature at which the heating isn't needed, and how much wind and sun change things.
2. **It's the yardstick for the experiments.** Once we know what the house *should* have burned, any change (the new blinds, a lower flow temperature) can be judged against that, so a mild week can't pass for a saving.

**ML ideas in this notebook:**
- **A baseline:** a model is only worth something if it beats a deliberately dumb guess.
- **Feature engineering:** turning raw weather into inputs that match the physics (degree-days, wind × cold).
- **A tuned setting:** the "balance temperature" is found by trying many values and keeping the best.
- **Honest testing:** holding out whole months, never random days.
- **Interpretation:** every number the model learns has a physical meaning we can sanity-check.
- **Residuals:** the days the model gets wrong are often the most interesting.

**Words used here:** see the glossary straight after the set-up cell (it's `GLOSSARY.md`, also readable on GitHub).

In [ ]:
import json, os, pathlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from wiser_ml import data, plots, signature as sg

plots.style()
pd.set_option("display.precision", 2)

# The real database: a copy of the collector's wiser.db (never committed).
# Without it, the notebook runs on a made-up house so it can be tried anywhere.
DB = pathlib.Path(os.environ.get("WISER_DB", "data/wiser.db"))
AWAY = pathlib.Path(os.environ.get("WISER_AWAY_DAYS", "data/away_days.txt"))   # optional list of trips
if DB.exists():
    daily_all, hourly = data.load_daily(DB, AWAY if AWAY.exists() else None)
    SOURCE = "real data from %s" % DB.name
else:
    daily_all, hourly = data.demo_daily()
    SOURCE = "MADE-UP DEMO DATA (no data/wiser.db found)"
print("Using", SOURCE)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(pathlib.Path("GLOSSARY.md").read_text()))

## 1. What we have to learn from

Gas and electricity come from the smart meter via the Bright app. Bright is missing whole days, so a day only counts if all 48 half-hours are there; we never guess the missing ones. Weather is Open-Meteo's hourly data.

The heating season is October to April. In summer the gas only heats water, which is a different question from heating the house, so the model learns from heating-season days only.

**Days away are left out.** The model predicts a *normal day at home*. A day with almost no gas (under 0.5 kWh, meaning no hot water at all) means nobody was home, so it's left out of training. Trips where some gas was still used (e.g. frost protection) can be listed in `data/away_days.txt`, one date or `YYYY-MM-DD to YYYY-MM-DD` per line.

In [ ]:
daily = data.heating_season(daily_all)
cov = data.coverage(daily_all)
cov

In [ ]:
away = daily_all[daily_all["away"].astype(bool)]
print(f"Days left out as away from home: {len(away)}" + (": " + ", ".join(d.strftime("%a %d %b %Y") for d in away.index) if len(away) else ""))
n = data.home_days(daily)["gas_kwh"].count()
print(f"Heating-season days with complete gas data: {n}")
print(f"Outdoor temperature range on those days: "
      f"{daily.loc[daily.gas_kwh.notna(), 'temp_mean_c'].min():.1f} to "
      f"{daily.loc[daily.gas_kwh.notna(), 'temp_mean_c'].max():.1f} °C (daily mean)")

### How much gas goes on hot water?

With a combi boiler, gas heats both the radiators and the hot water. On warm days (May to September, a daily average of 15 °C or more) the heating is off, so the gas used is **hot water** (plus the hob, if it's gas). We measure it directly from those days instead of asking the model to guess it, because the model's own guess turned out to be poor (see section 4).

In [ ]:
HW = data.hot_water_kwh(daily_all)
hw = HW["kwh"] if HW else None
if HW:
    print(f"Hot water on a normal day at home: about {HW['kwh']:.1f} kWh/day "
          f"(typically {HW['low']:.1f} to {HW['high']:.1f}; from {HW['days']} warm days with the heating off)")
else:
    print("Not enough warm days to measure hot water; the model will estimate it instead.")

## 2. The energy signature

Each dot is one day: its average outdoor temperature against the gas used. If the physics holds, we should see a **hockey stick**: a sloping line on cold days (heating), flattening out on warm days (hot water only). The flattening point is the **balance temperature**.

In [ ]:
d = data.home_days(daily).copy()
d["season"] = plots.season_of(d.index)
fig, ax = plt.subplots()
for season, colour in plots.SEASON_COLOURS.items():
    s = d[d.season == season]
    if len(s):
        ax.scatter(s.temp_mean_c, s.gas_kwh, s=30, color=colour, alpha=0.85,
                   edgecolor=plots.SURFACE, linewidth=0.8, label=f"{season} ({len(s)} days)")
ax.set(title="Daily gas use against outdoor temperature",
       xlabel="Daily mean outdoor temperature (°C)", ylabel="Gas used (kWh per day)")
ax.legend(loc="upper right")
plt.show()

## 3. Four models, simplest first

| Model | What it assumes |
|---|---|
| `mean` | Every day uses the average amount. The baseline to beat: it knows nothing about weather. |
| `hdd_15_5` | Gas rises with **heating degree-days** at the UK's conventional 15.5 °C base. A degree-day is one degree below the base for one day, summed hour by hour. |
| `changepoint` | The same, but the base (balance) temperature is **learned** from the data: we try every value from 10 to 20 °C and keep the best fit. |
| `full` | Change-point, plus **wind × degree-days** (draughts only cost heat when it's cold), **sunshine** (free heat through the windows) and **weekends** (more people at home). |

**How they're tested:** each month is held out in turn. The model is fitted on all the other months and asked to predict the held-out one. We don't hold out random days, because neighbouring days share the same weather spell, so a random split would be testing the model on days it has effectively already seen.

**The scores:**
- **MAE:** the typical miss, in kWh per day.
- **CV(RMSE):** the typical miss as a % of an average day, with big misses counted extra.
- **NMBE:** the overall bias in %, i.e. whether it systematically over- or under-predicts.

CV(RMSE) and NMBE are the measures ASHRAE Guideline 14 uses to judge building energy models.

In [ ]:
scores = sg.compare(daily, hourly, hot_water=hw)
scores[["days", "mae_kwh", "cv_rmse_pct", "nmbe_pct"]]

## 4. Inside the best model

**Hot water is fixed, not learned.** At first, the model learned the hot-water level itself (the "intercept"). It came out at 8–14 kWh a day, nothing like the 3.5 kWh actually measured on warm days. There are too few heating-season days near the balance point to pin it down, so it soaked up some mild-weather heating. Fixing it at the measured value is a standard move: use what you already know, and let the model learn only what it can't. It also gave the best held-out score, and left the weather effects unchanged, which suggests they're real.

The `full` model's learned numbers, each with a 95% confidence interval. The intervals use "HAC" standard errors, which allow for the fact that a cold spell lasts several days, so consecutive days aren't really independent evidence.

**Sanity checks, because every number has a physical meaning:**
- The degree-day slope must be **positive**: colder means more gas.
- Wind should **add** gas when it's cold.
- Sun should **save** gas.
- The balance temperature should be in the low-to-mid teens °C, the usual range for UK homes.

If any of these came out the wrong way, the model would be fitting noise, and we'd know not to trust it.

**One physical guard:** predictions never go below the hot-water level. Without it, the sunshine term pushed two mild, sunny April days below zero gas, which is impossible: sun can offset heating, not hot water. Scaling the sun's effect by how much heating each day needed was also tried, and rejected: it made the hot-water figure negative.

**The balance temperature and the hot-water level trade off against each other.** Fixing hot water lower moved the balance point up (to about 18–19 °C). Treat the balance temperature as approximate. The slope, wind, sun and weekend effects stayed the same either way, and they're what the experiments rely on.

In [ ]:
full = sg.fit(daily, hourly, "full", hot_water=hw)
ci = full.result.conf_int()
table = pd.DataFrame({"estimate": full.params, "low (95%)": ci[0], "high (95%)": ci[1],
                      "p-value": full.result.pvalues})
print(f"Balance temperature: {full.base_c:.2f} °C;  hot water fixed at {full.base_load:.1f} kWh/day")
table

In [ ]:
sse = pd.Series(full.sse_by_base)
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(sse.index, sse.values / sse.min(), color=plots.BLUE)
ax.axvline(full.base_c, color=plots.MUTED, linewidth=1, linestyle="--")
ax.annotate(f"best fit: {full.base_c:.2f} °C", (full.base_c, 1), textcoords="offset points",
            xytext=(8, 12), color=plots.INK)
ax.set(title="Choosing the balance temperature",
       xlabel="Balance temperature tried (°C)", ylabel="Error relative to best (1 = best)")
plt.show()

## 5. What it says about the house

The model's numbers, turned into plain statements.

The **heat-loss coefficient** is the whole house's heat loss in watts for each degree of difference between inside and outside. It assumes the combi boiler turns 88% of the gas into heat, so treat it as approximate. It's most useful for comparing before and after a change.

In [ ]:
p = full.params
slope = p["hdd"]
h = sg.heat_loss_w_per_k(slope)
typical_wind = daily.wind_mean_ms.median()
lines = [
    f"Hot water: about {full.base_load:.1f} kWh/day on a normal day at home (measured on warm days); none when away.",
    f"Heating starts below about {full.base_c:.1f} °C outside (daily average).",
    f"Each degree-day of cold costs about {slope:.1f} kWh of gas at typical wind ({typical_wind:.1f} m/s).",
    f"Heat-loss coefficient: about {h:.0f} W per °C inside-outside difference (assuming 88% boiler efficiency).",
    f"A day 10 °C below the balance point with 5 m/s more wind than usual costs about "
    f"{10 * 5 * p['hdd_x_wind']:.1f} kWh extra.",
    f"Each kWh/m² of sunshine saves about {-p['solar_kwh_m2']:.1f} kWh of gas.",
    f"Weekends use about {p['weekend']:+.1f} kWh/day compared with weekdays in the same weather.",
]
print("\n".join("• " + l for l in lines))

## 6. Where the model is wrong

**Residual** = actual gas minus what the model predicted for that day's weather (using the held-out-month predictions, so the model never saw the day it's judging).
- **A big positive residual:** the house used more gas than the weather explains. That could be a window left open, guests, or a heating override.
- **A big negative one:** it used less. That could be time away, the electric heaters doing some of the work, or a sunny day the model under-rated.

In [ ]:
overall, by_month, pred = sg.leave_one_month_out(daily, hourly, "full", hot_water=hw)
pred["residual"] = pred.actual - pred.predicted
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.axhline(0, color=plots.MUTED, linewidth=1)
ax.scatter(pred.index, pred.residual, s=24, color=plots.BLUE, edgecolor=plots.SURFACE, linewidth=0.6)
ax.set(title="How far off each day's prediction was (held-out months)",
       ylabel="Actual minus predicted (kWh)", xlabel="")
plt.show()

In [ ]:
cols = ["actual", "predicted", "residual"]
extra = daily_all.loc[pred.index, ["temp_mean_c", "wind_mean_ms", "solar_kwh_m2", "elec_kwh", "weekend"]]
worst = pred[cols].join(extra)
print("Days that used MORE gas than the weather explains:")
display(worst.sort_values("residual", ascending=False).head(8).round(1))
print("Days that used LESS gas than the weather explains:")
display(worst.sort_values("residual").head(8).round(1))

**The Conservatory's electric heaters:** on really cold days they add heat that the gas model doesn't know about. If they matter, high-electricity days should sit below the line (less gas than expected). The correlation below checks that. Negative means more electricity goes with less gas than predicted.

In [ ]:
both = worst.dropna(subset=["elec_kwh"])
r = both[["residual", "elec_kwh"]].corr().iloc[0, 1]
print(f"Correlation between gas residual and electricity used: {r:+.2f} ({len(both)} days)")

## 7. How well each held-out month was predicted

In [ ]:
by_month[["days", "mae_kwh", "cv_rmse_pct", "nmbe_pct"]]

## 8. Save the yardstick

The fitted model is saved so the blinds experiment (TECH-104) and the flow-temperature experiment (TECH-105) can ask what the house *would* have used for the weather on any given day. The file goes to `output/`, which isn't committed.

In [ ]:
pathlib.Path("output").mkdir(exist_ok=True)
saved = {
    "model": "full", "source": SOURCE, "balance_c": full.base_c, "hot_water_kwh": full.base_load,
    "params": {k: float(v) for k, v in full.params.items()},
    "trained_on": {"first_day": str(data.home_days(daily).index.min().date()),
                   "last_day": str(data.home_days(daily).index.max().date()),
                   "days": int(len(data.home_days(daily)))},
    "held_out_scores": {k: float(v) for k, v in overall.items()},
}
pathlib.Path("output/energy_signature.json").write_text(json.dumps(saved, indent=2))
print(json.dumps(saved, indent=2))

## 9. Caveats

- **This is the whole house, not room by room.** Per-room heat loss is the next notebook (the RC model), which needs the floor plan from TECH-102.
- **Gas includes hot water.** It's a combi boiler, so showers and baths add day-to-day noise that the weather can't explain. Hot water is fixed at its typical level, but real days vary, and it's zero on days away.
- **Days away are only partly caught.** Days with no gas at all are left out automatically. Trips where some gas was still used need listing in `data/away_days.txt`, or they look like unusually efficient days.
- **Missing days are skipped, never guessed,** so months with fewer complete days carry less weight.
- **Days are UTC days.** In summer that's an hour off UK time, which barely changes daily totals.
- **The heat-loss coefficient assumes 88% boiler efficiency.** It's for comparing before and after, not for comparing with other houses.
- **The start of each heating season is fuzzy.** In early October the heating may not be switched on yet, so the model, which assumes it is, over-predicts. October 2025 had only 7 complete days and is the worst-predicted month. Room data (from August 2026) can mark when the heating was actually running, for this winter only.
- **The house isn't the same every year.** It changes between winters (the blinds, and anything else done), so a model trained across both winters averages over those changes. The experiments handle this by comparing against the model trained before each change.